# Doyle Exercise 4.15: forward simulation

Follow the same six steps as the other examples. Only mechanism.json and thermo.json are needed to simulate the process. The nominal textbook input is an equilibrium; edit the feed flow in Step 2 to see a transient. These are textbook model predictions, not experimental validation. Independent numerical qualification is available separately through run_benchmarks.py.


## Step 1: Select input and output folders


In [ ]:
# STEP 1: Select input and output folders
# Run cells in order; change example_dir for a new process.
# Exports replace matching files in EXPORT_DIR.
from pathlib import Path
import csv
import json
import sys
import numpy as np
import matplotlib.pyplot as plt

repo_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                  if (path / 'PharmaPy').is_dir()), None)
if repo_root is None:
    raise ValueError('Set repo_root to your PharmaPy checkout before running this cell')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from PharmaPy.Bioreactors import build_bioreactor

example_dir = repo_root / 'examples/bioreactors/Doyle_Exercise_4_15'
input_dir = example_dir / 'inputs'
thermo_path = input_dir / 'thermo.json'
EXPORT_DIR = example_dir / 'outputs'
WRITE_ARTIFACTS = True
# Choose another output folder to preserve previous exports.


## Step 2: Load and edit forward-simulation inputs


In [ ]:
# STEP 2: Load and edit forward-simulation inputs
# The two JSON files fully specify this forward model.
# Biological parameters and units belong in the explicit inputs.
config = json.loads((input_dir / 'mechanism.json').read_text())
# Edit the JSON files or the loaded mapping; rerun subsequent cells after edits.
# For the +10% dilution step, uncomment this input change:
# config['recipes']['continuous'][0]['volume_l_h'] = 0.11  # [L/h]
# At the declared 1 L volume this gives dilution 0.11/h, versus nominal 0.1/h.
print(json.dumps(config, indent=2))


## Step 3: Run the native forward simulation


In [ ]:
# STEP 3: Run the native forward simulation
# Construct a fresh native reactor from the input mapping.
# Solve its configured operating schedule without a benchmark runner.
assembly = build_bioreactor(config, thermo_path=thermo_path)
unit = assembly.unit
histories = assembly.solve(verbose=False)
print('Native reactor:', type(unit).__name__)
print('Recorded segments:', len(histories))


## Step 4: Build the trajectory table


In [ ]:
# STEP 4: Build the trajectory table
# Convert native inventories using each recorded reactor volume.
# Biomass is a configured lumped mass species, not the separate biomass state.
species = tuple(config['state']['species']['amounts'])
positions = [list(assembly.phase.name_species).index(name) for name in species]
rows = []
for history in histories:
    time_h = np.asarray(history.time).reshape(-1) / 3600.0  # [h], from native seconds
    volume_l = np.asarray(history.vessel_vol).reshape(-1) * 1000.0  # [L], from m3
    amounts_g = np.asarray(history.mass_j_liquid0)[:, positions] * 1000.0  # [g], from kg
    concentrations = amounts_g / volume_l[:, None]  # [g/L]
    for time, volume, values in zip(time_h, volume_l, concentrations):
        row = dict(time_h=float(time), liquid_volume_l=float(volume))
        row.update({name + '_g_l': float(value) for name, value in zip(species, values)})
        rows.append(row)
print('Initial row:', rows[0])
print('Final row:', rows[-1])


## Step 5: Plot the simulated trajectories


In [ ]:
# STEP 5: Plot the simulated trajectories
# Plot the same rows used for the exported table.
# Species selection and labels are explicit and editable here.
figure, axes = plt.subplots(len(species), 1, figsize=(8, 7), sharex=True, squeeze=False, constrained_layout=True)
for axis, name in zip(axes[:, 0], species):
    axis.plot([row['time_h'] for row in rows], [row[name + '_g_l'] for row in rows], label=name)
    axis.set(ylabel=name + ' [g/L]')
    axis.grid(alpha=0.25)
    axis.legend()
axes[0, 0].set(title='Doyle Exercise 4.15 forward simulation')
axes[-1, 0].set(xlabel='Time [h]')
plt.show()


## Step 6: Export the trajectory spreadsheet and figures


In [ ]:
# STEP 6: Export the trajectory spreadsheet and figures
# Save a spreadsheet-compatible CSV plus PNG and SVG figures.
# Exports replace matching files; no verification runner or parameter fitting is called.
if WRITE_ARTIFACTS:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    with (EXPORT_DIR / 'trajectories.csv').open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]), lineterminator='\n')
        writer.writeheader()
        writer.writerows(rows)
    figure.savefig(EXPORT_DIR / 'trajectories.png', dpi=180)
    figure.savefig(EXPORT_DIR / 'trajectories.svg')
    print('Saved trajectories.csv, trajectories.png and trajectories.svg to', EXPORT_DIR)
else:
    print('Export disabled; trajectories and figure remain in memory.')
